In [32]:
import plotly.express as px
import pandas as pd
import numpy as np
from ipywidgets import interact, widgets
import plotly.graph_objects as go


In [2]:
data = pd.read_excel("./dados/T3.07.xls", skiprows=6)
data

,Tipo de Unidade,Unidade,Semestre,Métrica,Total,Total (%)
0,A - Ensino e Pesquisa,EACH,1,Aprovação,19375,88.02
1,A - Ensino e Pesquisa,EACH,1,Matriculados,22011,100.00
2,A - Ensino e Pesquisa,EACH,1,Reprovação por Frequência e Nota,1357,6.16
3,A - Ensino e Pesquisa,EACH,1,Reprovação por Frequência,32,0.14
4,A - Ensino e Pesquisa,EACH,1,Reprovação por Nota,1247,5.66
...,...,...,...,...,...,...
585,G - Programas Conjuntos,Pró-G,2,Aprovação,8500,88.67
586,G - Programas Conjuntos,Pró-G,2,Matriculados,9586,100.00
587,G - Programas Conjuntos,Pró-G,2,Reprovação por Frequência e Nota,916,9.55
588,G - Programas Conjuntos,Pró-G,2,Reprovação por Frequência,110,1.14


In [3]:
data['Métrica'].value_counts()

Métrica
Aprovação                           118
Matriculados                        118
Reprovação por Frequência e Nota    118
Reprovação por Frequência           118
Reprovação por Nota                 118
Name: count, dtype: int64

In [4]:
data_aprovacao = data[data['Métrica'] == 'Aprovação']
media_aprovacao = data_aprovacao.groupby("Unidade")["Total (%)"].mean().reset_index().sort_values(by='Total (%)', ascending = False)
media_aprovacao

,Unidade,Total (%)
27,FMVZ,99.205
4,EE,99.200
29,FOB,98.780
30,FORP,98.655
26,FMRP,98.110
28,FO,97.760
57,MZ,97.760
24,FM,97.715
8,EERP,97.645
35,IAU,97.560


In [5]:
# Filtra apenas as linhas de reprovação
reprovacoes = data[data["Métrica"].str.contains("Reprovação", case=False)]

# Soma o total de reprovações por unidade
total_reprov = (
    reprovacoes.groupby("Unidade")["Total"]
    .sum()
    .reset_index()
    .rename(columns={"Total": "Total Reprovações"})
)

# Junta com os motivos detalhados via pivot
motivos = (
    reprovacoes.pivot_table(
        index="Unidade",
        columns="Métrica",
        values="Total (%)"
    )
    .reset_index()
)

# Junta os dois resultados
resultado = pd.merge(total_reprov, motivos, on="Unidade", how="left")

# Ordena do maior para o menor total de reprovação
resultado = resultado.sort_values(by="Total Reprovações", ascending=False)

resultado.head()


,Unidade,Total Reprovações,Reprovação por Frequência,Reprovação por Frequência e Nota,Reprovação por Nota
11,EP,9110,0.090,7.875,11.320
45,IME,6624,0.045,9.985,16.280
23,FFLCH,6443,0.150,7.060,3.855
2,EACH,4675,0.120,6.005,5.170
7,EEL,4472,0.070,7.050,15.530


In [41]:
def plot_reprovacoes_interativo(
    motivos, 
    top_n=10, 
    ordem="maior", 
    titulo="Taxa de Reprovação(%) por Unidade e Motivo"
):
    """
    Gera um gráfico de barras empilhadas horizontais (laterais) interativo
    com motivos de reprovação por unidade.
    """

    # Paleta fixa (Okabe-Ito)
    palette = ['#0072B2', '#E69F00', '#000000']

    # Total de reprovação
    motivos["Total Reprovação (%)"] = motivos.drop(columns="Unidade").sum(axis=1)
    ascending = True if ordem == "menor" else False
    motivos_ordenado = motivos.sort_values(by="Total Reprovação (%)", ascending=ascending)
    motivos_filtrado = motivos_ordenado.head(top_n).drop(columns="Total Reprovação (%)")
    motivos_filtrado = motivos_filtrado[::-1]


    # Adequando formato de análise
    df_melt = motivos_filtrado.melt(
        id_vars="Unidade",
        var_name="Motivo",
        value_name="Reprovação (%)"
    )

    # Simplificando legenda
    df_melt["Motivo Completo"] = df_melt["Motivo"].replace({
        "Reprovação por Frequência e Nota": "Frequência e Nota",
        "Reprovação por Frequência": "Frequência",
        "Reprovação por Nota": "Nota"
    })

    # Ordem das legendas
    ordem_motivos = ["Frequência e Nota", "Nota", "Frequência"]
    df_melt["Motivo Completo"] = pd.Categorical(
        df_melt["Motivo Completo"],
        categories=ordem_motivos,
        ordered=True
    )
    # ordem_motivos = ordem_motivos[::-1]
    # Gráfico de barras empilhadas HORIZONTAIS
    fig = px.bar(
        df_melt,
        y="Unidade",                 # eixo Y -> categorias
        x="Reprovação (%)",          # eixo X -> valores numéricos
        color="Motivo Completo",
        category_orders={"Motivo Completo": ordem_motivos},
        color_discrete_sequence=palette,
        barmode="stack",
        orientation="h",             # <-- chave para barras horizontais
        title=titulo
    )

    # Reordenação e cores
    ordem_trace = ["Frequência", "Nota", "Frequência e Nota"]
    traces_ordenados = [trace for nome in reversed(ordem_trace) for trace in fig.data if trace.name == nome]
    for i, trace in enumerate(traces_ordenados):
        trace.marker.color = palette[i]
        trace.name = f"<span style='color:{palette[i]}'>{trace.name}</span>"
    fig.data = traces_ordenados

    # Layout
    fig.update_layout(
        plot_bgcolor="white",
        xaxis=dict(showgrid=False, color="gray", linecolor="gray", title="Taxa de Reprovação (%)"),
        yaxis=dict(showgrid=False, color="gray", linecolor="gray", title="Unidade"),
        legend=dict(
            title="Motivo",
            x=1.02, y=1,
            bordercolor="gray",
            borderwidth=1,
            font=dict(size=12),
            itemsizing="trace"
        ),
        legend_traceorder="reversed",
        hovermode="y unified",
    )

    
    # Tooltip
    fig.update_traces(
        hovertemplate="%{fullData.name}: %{x:.2f}%<extra></extra>",
        marker_line_color="black",
        marker_line_width=0.4,
        text=None
    )

    fig.show()


In [42]:
plot_interativo_widget(motivos)


interactive(children=(IntSlider(value=10, description='Top N', max=30, min=1), Dropdown(description='Ordem', o…

## Análises Plot Interativo
- Objetivo:
  - Compreender como a taxa de reprovação se distribui por unidade e por motivo de reprovação (Frequência, Nota ou ambos)
- Perguntas:
  - Quais unidades com maior taxa de reprovação? E com menor?
  - Quais os principais motivos de reprovação dessas unidades com menor/maior reprovação
- Justificativa de Escolha de Visualização
  - Pela leitura em Z, o usuário consegue ter a informação mais destacada no canto superior direito, por isso a informação vem como barras horizontais empilhadas, e não como barras verticais empilhadas (a informação de maior importância está no eixo y)
  - Por ser barras empilhadas, o usuário tem mais informação do que apenas uma coloração na barra.
  - O usuário pode ter compreensão de que, por exemplo, o IME tem a maior taxa de reprovação, mas consegue compreender também que o motivo mais destacado é a Nota. Além disso, consegue perceber que reprovação exclusivamente por frequência não é comum, fatos que seriam impossíveis de visualizar em gráfico de barras tradicional
  - Nesse contexto, com 2 categóricas e 1 numérica, o gráfico de barras empilhadas horizontais é justifica-se por esses fatores, e além disso é interativo. Por frequência ser a menor classe, fica difícil comparar só olhando qual seriam as menores reprovações por frequência dentre as 10 maiores reprovações, mas isso pode ser facilmente visualidado apenas ao tirar a seleção das outras duas categorias na legenda
  - Além disso, a biblioteca de Ipywidgets deixa a visualização mais interativa com um slider das top N informações vistas, e a ordem (maiores valores ou menores valores), e altera automaticamente o gráfico.

- Insights:
  - IME, MAC e EEL são, respectivamente, as três unidades com maior reprovação total
  - Dentre as 10 maiores reprovações totais, os com maior taxa de reprovação por frequência são MAC, IFSC e Interunidades Licenciatura São Carlos
  - "Nota e Frequência" e "Nota" competem para principais motivos de reprovação. Note que exclusivamente frequência é um motivo bem comum, o que pode levantar hipóteses de que se alunos reprovam por frequência, há uma tendência de que também reprovem por nota.
  - As menores taxas de reprovação são, do menor pro maior, "EESC e ICMC", "FCF e ICB", "FMVZ", e para ICMC e EESC o motivo é exclusivamente frequência e nota, enquanto para FCF e ICB a maioria pe por nota.

    